# Ollama + Qwen для Cline (второй аккаунт Colab)

Этот блокнот делает из Colab с видеокартой сервер модели для агента Cline. Сам Cline работает в общей рабочей области `colab-sandbox` (там же, где Claude и Cline на DeepSeek) и ходит к модели через туннель.

1. **Среда выполнения → Сменить среду выполнения → A100** (лучше) или **L4**. На T4 будет только 14B-модель.
2. **Среда выполнения → Выполнить все.**
3. В конце ячейка 5 печатает **Base URL** и **API Key** для Cline.

Что изменено по сравнению с исходным планом:
- модель выбирается по памяти видеокарты: `qwen3-coder:30b` (MoE, активны ~3 млрд параметров, быстрая и надёжно вызывает инструменты Cline) на A100/L4;
- модели хранятся на Диске, но при старте копируются на локальный диск (читать 19 ГБ с Диска при каждом запуске очень медленно), новые сохраняются обратно в фоне;
- Ollama слушает только `127.0.0.1`; наружу через туннель смотрит маленький прокси, который пропускает только запросы с ключом (без ключа любой, кто узнал ссылку, пользуется вашей видеокартой);
- ключ хранится на Диске (или в Colab Secrets `OLLAMA_PROXY_KEY`) и не меняется при перезапуске, меняется только ссылка туннеля;
- вывод `cloudflared` читается в фоне (в исходном коде его переставали читать, и туннель мог зависнуть, когда буфер заполнится);
- параметры генерации взяты из карточки Qwen3-Coder; системный промпт в модели не задаётся: правила проекта лежат в `.clinerules` репозитория;
- ячейки с собственным агентом и GitHub-токеном убраны: PR и задачи делает Cline по правилам репозитория, токен в этот блокнот не вводится.

In [ ]:
# 1. Диск, видеокарта и выбор модели
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/OpenBFME_Ennorath_AI'
DRIVE_MODELS = WORK + '/ollama_models'
LOCAL_MODELS = '/content/ollama_models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)

q = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True)
if q.returncode != 0:
    raise SystemExit('Нет видеокарты: Среда выполнения -> Сменить среду выполнения -> A100 (лучше) или L4')
GPU_NAME, mem = [x.strip() for x in q.stdout.strip().splitlines()[0].split(',')]
VRAM_GB = int(mem) / 1024

# Модель под память видеокарты (веса Q4_K_M):
#   A100 40/80 ГБ: qwen3-coder:30b, контекст 64K
#   L4 24 ГБ:      qwen3-coder:30b, контекст 32K (64K вместе с весами не помещается)
#   T4 15 ГБ:      qwen2.5-coder:14b, контекст 32K (30B не помещается)
if VRAM_GB >= 38:
    BASE_MODEL, CTX = 'qwen3-coder:30b', 65536
elif VRAM_GB >= 22:
    BASE_MODEL, CTX = 'qwen3-coder:30b', 32768
else:
    BASE_MODEL, CTX = 'qwen2.5-coder:14b', 32768
AGENT_MODEL = 'ennorath-coder'
print(f'Видеокарта: {GPU_NAME}, {VRAM_GB:.0f} ГБ -> {BASE_MODEL}, контекст {CTX}')

In [ ]:
# 2. Установка Ollama, cloudflared, aiohttp; скачанные раньше модели с Диска на локальный диск
!apt-get -qq update > /dev/null && apt-get install -y -qq zstd rsync > /dev/null
!command -v ollama > /dev/null || (curl -fsSL https://ollama.com/install.sh | sh > /dev/null)
!test -x /usr/local/bin/cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)
!pip install -q aiohttp
!rsync -a "{DRIVE_MODELS}/" "{LOCAL_MODELS}/" && du -sh "{LOCAL_MODELS}"

In [ ]:
# 3. Ollama только на 127.0.0.1 и модель ennorath-coder с нужным контекстом
import time, urllib.request
env = dict(os.environ,
    OLLAMA_MODELS=LOCAL_MODELS, OLLAMA_HOST='127.0.0.1:11434',
    OLLAMA_FLASH_ATTENTION='1', OLLAMA_KV_CACHE_TYPE='q8_0',
    OLLAMA_KEEP_ALIVE='24h', OLLAMA_CONTEXT_LENGTH=str(CTX), OLLAMA_NUM_PARALLEL='1')
subprocess.run(['pkill', '-x', 'ollama'])
ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit('Ollama не запустилась, смотрите /content/ollama.log')

subprocess.run(['ollama', 'pull', BASE_MODEL], env=env, check=True)
lines = ['FROM ' + BASE_MODEL, f'PARAMETER num_ctx {CTX}']
if BASE_MODEL.startswith('qwen3-coder'):
    # рекомендованные параметры из карточки Qwen3-Coder
    lines += ['PARAMETER temperature 0.7', 'PARAMETER top_p 0.8', 'PARAMETER top_k 20', 'PARAMETER repeat_penalty 1.05']
else:
    lines += ['PARAMETER temperature 0.2']
with open('/content/Modelfile_ennorath', 'w') as f:
    f.write(chr(10).join(lines) + chr(10))
subprocess.run(['ollama', 'create', AGENT_MODEL, '-f', '/content/Modelfile_ennorath'], env=env, check=True)
# скачанное сохраняется на Диск в фоне: в следующий раз модель не качается заново
subprocess.Popen(['rsync', '-a', LOCAL_MODELS + '/', DRIVE_MODELS + '/'])
print(subprocess.run(['ollama', 'list'], env=env, capture_output=True, text=True).stdout)

# прогрев напрямую, мимо туннеля: модель грузится в видеопамять один раз. Туннель Cloudflare обрывает запрос,
# на который нет ответа 100 секунд (ошибка 524), а первая загрузка 19 ГБ дольше
import json
t0 = time.time()
warm = json.dumps({'model': AGENT_MODEL, 'prompt': 'OK', 'stream': False, 'options': {'num_predict': 1}}).encode()
urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:11434/api/generate', data=warm, headers={'Content-Type': 'application/json'}), timeout=1800).read()
print(f'Модель в видеопамяти, загрузка {time.time() - t0:.0f} с')

In [ ]:
%%writefile /content/ollama_proxy.py
# 4. Прокси перед Ollama: пропускает только запросы с ключом (Authorization: Bearer <ключ>),
# ответ модели передаёт потоком (Cline показывает текст по мере генерации).
import hmac, os
from aiohttp import web, ClientSession, ClientTimeout

KEY = os.environ['PROXY_KEY']
UPSTREAM = 'http://127.0.0.1:11434'
DROP = {'host', 'authorization', 'content-length', 'transfer-encoding', 'connection'}

async def handle(request):
    auth = request.headers.get('Authorization', '')
    token = auth[7:] if auth.startswith('Bearer ') else request.headers.get('X-Api-Key', '')
    if not hmac.compare_digest(token.encode(), KEY.encode()):
        return web.Response(status=401, text='unauthorized')
    headers = {k: v for k, v in request.headers.items() if k.lower() not in DROP}
    body = await request.read()
    async with ClientSession(timeout=ClientTimeout(total=None)) as session:
        async with session.request(request.method, UPSTREAM + request.rel_url.path_qs, headers=headers, data=body) as up:
            resp = web.StreamResponse(status=up.status)
            if up.headers.get('Content-Type'):
                resp.headers['Content-Type'] = up.headers['Content-Type']
            await resp.prepare(request)
            async for chunk in up.content.iter_any():
                await resp.write(chunk)
            await resp.write_eof()
            return resp

app = web.Application(client_max_size=256 * 1024 * 1024)
app.router.add_route('*', '/{tail:.*}', handle)
web.run_app(app, host='127.0.0.1', port=11435, print=None)

In [ ]:
# 5. Ключ, прокси, туннель Cloudflare и проверка
import secrets, re, threading, json, urllib.error
KEY_FILE = WORK + '/proxy_key.txt'
PROXY_KEY = None
try:
    from google.colab import userdata
    PROXY_KEY = userdata.get('OLLAMA_PROXY_KEY')
except Exception:
    pass
if not PROXY_KEY:
    if os.path.exists(KEY_FILE):
        PROXY_KEY = open(KEY_FILE).read().strip()
    else:
        PROXY_KEY = 'ok-' + secrets.token_urlsafe(24)
        open(KEY_FILE, 'w').write(PROXY_KEY)

subprocess.run(['pkill', '-f', 'ollama_proxy.py'])
subprocess.run(['pkill', '-x', 'cloudflared'])
proxy_proc = subprocess.Popen(['python3', '/content/ollama_proxy.py'], env=dict(os.environ, PROXY_KEY=PROXY_KEY),
                              stdout=open('/content/proxy.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(2)
cf = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11435', '--no-autoupdate'],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
def _drain():
    global PUBLIC_URL
    with open('/content/cloudflared.log', 'w') as log:
        for line in cf.stdout:
            log.write(line)
            log.flush()
            m = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com', line)
            if m and not PUBLIC_URL:
                PUBLIC_URL = m.group(0)
threading.Thread(target=_drain, daemon=True).start()
for _ in range(60):
    if PUBLIC_URL:
        break
    time.sleep(1)
if not PUBLIC_URL:
    raise SystemExit('Туннель не поднялся, смотрите /content/cloudflared.log')

def call(path, key=None, body=None):
    headers = {'Content-Type': 'application/json'}
    if key:
        headers['Authorization'] = 'Bearer ' + key
    return urllib.request.urlopen(urllib.request.Request(PUBLIC_URL + path, data=body, headers=headers), timeout=600)

for _ in range(30):  # новый адрес туннеля появляется в DNS не сразу
    try:
        call('/v1/models')
        print('ВНИМАНИЕ: без ключа ответил, прокси не работает')
        break
    except urllib.error.HTTPError as e:
        print('Запрос без ключа:', e.code, '(так и надо: 401)')
        break
    except Exception:
        time.sleep(2)
req = json.dumps({'model': AGENT_MODEL, 'messages': [{'role': 'user', 'content': 'Reply with one word: OK'}], 'max_tokens': 5}).encode()
for attempt in range(5):  # 524: модель ещё грузится (ответа нет 100 с), повторить
    try:
        print('Запрос с ключом:', json.load(call('/v1/chat/completions', PROXY_KEY, req))['choices'][0]['message']['content'])
        break
    except urllib.error.HTTPError as e:
        if e.code != 524 or attempt == 4:
            raise
        print('524: модель ещё грузится, повтор через 30 с')
        time.sleep(30)
print('Если Cline однажды получит 524: это тот же обрыв туннеля через 100 с; нажать Retry.')

print('=' * 70)
print('Настройки Cline (провайдер OpenAI Compatible):')
print('  Base URL:   ' + PUBLIC_URL + '/v1')
print('  API Key:    ' + PROXY_KEY)
print('  Model ID:   ' + AGENT_MODEL)
print(f'  Context Window Size: {CTX}')
print('Ключ постоянный; при каждом перезапуске меняется только Base URL.')
print('=' * 70)

In [ ]:
# 6. Состояние (запускать когда угодно): модель в памяти, видеокарта, хвосты журналов
print(subprocess.run(['ollama', 'ps'], env=env, capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
!tail -n 5 /content/ollama.log /content/proxy.log /content/cloudflared.log

## Важно

- Вкладку с блокнотом держите открытой: Colab отключает простаивающую среду, и тогда пропадает и модель, и туннель. После перезапуска: «Выполнить все» и новый Base URL в Cline (ключ прежний).
- Ключ никому не показывать и не коммитить. Если он утёк: удалить `proxy_key.txt` на Диске и перезапустить ячейку 5.
- Файлы игры и game.dat в этот аккаунт не нужны: здесь только модель.